# ABC atlas taxonomy

Register the shared `CCN20230722` hierarchy from ABC metadata release `20230830`, with class, subclass, supertype and cluster colors.
This notebook writes taxonomy metadata only; cell mappings belong in cohort ETLs.

In [1]:
import hashlib
import json
from io import BytesIO
from pathlib import Path
from shutil import copyfile
from urllib.request import urlopen

import pandas as pd
import polars as pl

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import write_models
from connects_common_connectivity.models import (
    AlgorithmRun,
    Cluster,
    ClusterHierarchy,
    HierarchyCategory,
)


In [2]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'pyproject.toml').is_file()
)
INPUT_ROOT = REPO_ROOT / 'data/BARseq/extracted'
OUTPUT_ROOT = get_settings().output_root
HIERARCHY_ID = 'CCN20230722'
RELEASE = '20230830'
RUN_ID = 'abc_ccn20230722_20230830_import'
ROOT_ID = 'cell'
TREE_PATH = INPUT_ROOT / 'cluster_to_cluster_annotation_membership_pivoted.csv'
TERMS_PATH = INPUT_ROOT / 'cluster_annotation_term.csv'
SOURCE_TREE = (
    REPO_ROOT / 'data/BARseq/scripts/data/ref/abc_atlas/metadata' / TREE_PATH.name
)
TERMS_URL = (
    'https://allen-brain-cell-atlas.s3.us-west-2.amazonaws.com/'
    f'metadata/WMB-taxonomy/{RELEASE}/cluster_annotation_term.csv'
)
LEVELS = ['class', 'subclass', 'supertype', 'cluster']
EXPECTED_COUNTS = {'class': 34, 'subclass': 338, 'supertype': 1201, 'cluster': 5322}
print('Input:', INPUT_ROOT, '\nOutput:', OUTPUT_ROOT)
print('Taxonomy:', HIERARCHY_ID, 'Release:', RELEASE)


Input: /workspaces/ConnectsCommonConnectivity/data/BARseq/extracted 
Output: /workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7
Taxonomy: CCN20230722 Release: 20230830


## Cache reference metadata

Reuse saved files; copy the packaged tree or download the release-specific term/color table only when missing.

In [3]:
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
if not TREE_PATH.is_file():
    assert SOURCE_TREE.is_file(), f'Missing ABC parent-child table: {SOURCE_TREE}'
    copyfile(SOURCE_TREE, TREE_PATH)

if not TERMS_PATH.is_file():
    with urlopen(TERMS_URL, timeout=60) as response:
        content = response.read()
    # Validate the response before caching it; a failed download must not become a cache hit.
    downloaded = pd.read_csv(BytesIO(content), keep_default_na=False)
    assert {'label', 'name', 'color_hex_triplet'} <= set(downloaded.columns)
    assert downloaded['cluster_annotation_term_set_label'].str.startswith(
        HIERARCHY_ID + '_'
    ).all()
    TERMS_PATH.write_bytes(content)

tree = pd.read_csv(TREE_PATH, keep_default_na=False)
terms = pd.read_csv(TERMS_PATH, keep_default_na=False).rename(columns={
    'cluster_annotation_term_set_name': 'category', 'color_hex_triplet': 'hex_color',
})
for path, frame in ((TREE_PATH, tree), (TERMS_PATH, terms)):
    print(path.name, frame.shape)
    print(frame.head(3))


cluster_to_cluster_annotation_membership_pivoted.csv (5322, 6)
   cluster_alias neurotransmitter          class                subclass  \
0              1             Glut  01 IT-ET Glut  018 L2 IT PPP-APr Glut   
1              2             Glut  01 IT-ET Glut  018 L2 IT PPP-APr Glut   
2              3             Glut  01 IT-ET Glut  018 L2 IT PPP-APr Glut   

                   supertype                    cluster  
0  0082 L2 IT PPP-APr Glut_3  0326 L2 IT PPP-APr Glut_3  
1  0082 L2 IT PPP-APr Glut_3  0327 L2 IT PPP-APr Glut_3  
2  0081 L2 IT PPP-APr Glut_2  0322 L2 IT PPP-APr Glut_2  
cluster_annotation_term.csv (6905, 9)
                  label  name cluster_annotation_term_set_label  \
0  CS20230722_NEUR_Glut  Glut                  CCN20230722_NEUR   
1    CS20230722_NEUR_NA    NA                  CCN20230722_NEUR   
2  CS20230722_NEUR_GABA  GABA                  CCN20230722_NEUR   

  parent_term_label parent_term_set_label  term_set_order  term_order  \
0                   

## Shape hierarchy and colors

Check the packaged tree against the term table's authoritative parent IDs and use `color_hex_triplet` at every level.
Level-prefixed names distinguish identically named nodes; neurotransmitter annotations are not an additional tree level.

In [4]:
terms = terms.loc[terms['category'].isin(LEVELS)].sort_values(['term_set_order', 'term_order'])
assert terms.groupby('category').size().to_dict() == EXPECTED_COUNTS
assert terms['label'].is_unique and not terms.duplicated(['category', 'name']).any()
assert terms['hex_color'].str.fullmatch(r'#[0-9a-fA-F]{6}').all()
assert len(tree) == tree['cluster'].nunique() == 5322
term_sets = dict(zip(LEVELS, ['CLAS', 'SUBC', 'SUPT', 'CLUS']))
for category in LEVELS:
    subset = terms.loc[terms['category'].eq(category)]
    assert subset['cluster_annotation_term_set_label'].eq(
        f'{HIERARCHY_ID}_{term_sets[category]}'
    ).all()
    assert set(tree[category]) == set(subset['name'])
assert terms.loc[terms['category'].eq('class'), 'parent_term_label'].eq('').all()

for parent, child in zip(LEVELS, LEVELS[1:]):
    pairs = tree[[child, parent]].drop_duplicates()
    assert pairs[child].is_unique, f'{child} has multiple parents'
    parent_terms = terms.loc[terms['category'].eq(parent)].set_index('name')['label']
    child_terms = terms.loc[terms['category'].eq(child)]
    expected = child_terms['name'].map(pairs.set_index(child)[parent]).map(parent_terms)
    assert expected.eq(child_terms['parent_term_label']).all(), f'{child} parent mismatch'

# Prefixes prevent the shared supertype/cluster name from becoming a self-parent link.
terms['node_id'] = terms['category'] + ':' + terms['name']
node_by_label = dict(zip(terms['label'], terms['node_id']))
parent_of = {ROOT_ID: None}
parent_of.update({
    row.node_id: node_by_label[row.parent_term_label] if row.parent_term_label else ROOT_ID
    for row in terms.itertuples(index=False)
})
children_of = {node_id: [] for node_id in parent_of}
for node_id, parent in parent_of.items():
    if parent is not None:
        children_of[parent].append(node_id)

# Category levels count upward from leaves; node levels count downward from the root.
categories = [
    HierarchyCategory(id=category, hierarchy_id=HIERARCHY_ID, level=level)
    for level, category in enumerate([*reversed(LEVELS), 'major_class'])
]
depth = {category: level for level, category in enumerate(LEVELS, start=1)}
clusters = [Cluster(
    id=ROOT_ID, hierarchy_id=HIERARCHY_ID, children=children_of[ROOT_ID],
    level=0, hierarchy_category='major_class', hex_color='#000000',
)]
clusters.extend(
    Cluster(
        id=row.node_id, hierarchy_id=HIERARCHY_ID, parent=parent_of[row.node_id],
        children=children_of[row.node_id], level=depth[row.category],
        hierarchy_category=row.category, hex_color=row.hex_color,
    )
    for row in terms.itertuples(index=False)
)
assert len(clusters) == len(parent_of) == 6896
hierarchy = ClusterHierarchy(
    id=HIERARCHY_ID, run=RUN_ID, root=ROOT_ID, clusters=list(parent_of),
)
run = AlgorithmRun(
    id=RUN_ID, algorithm_name='ABC atlas taxonomy import', algorithm_version=RELEASE,
    json_object=json.dumps({
        'taxonomy_id': HIERARCHY_ID, 'metadata_release': RELEASE, 'terms_url': TERMS_URL,
        'sources': {
            path.name: hashlib.sha256(path.read_bytes()).hexdigest()
            for path in (TREE_PATH, TERMS_PATH)
        },
        'source_term_labels': dict(zip(terms['node_id'], terms['label'])),
    }),
)
print(terms.groupby('category').size())


category
class          34
cluster      5322
subclass      338
supertype    1201
dtype: int64


## Write shared taxonomy

In [5]:
for rows in ([run], categories, clusters, [hierarchy]):
    print(write_models(rows, output_root=OUTPUT_ROOT))


WrittenResult(class_name='AlgorithmRun', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/algorithmrun'), mode='merge_scoped', predicates=('target."id" = source."id"',), rows_written=1)


WrittenResult(class_name='HierarchyCategory', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/hierarchycategory'), mode='merge_scoped', predicates=('target."hierarchy_id" = source."hierarchy_id" AND target."id" = source."id" AND target."hierarchy_id" IN (\'CCN20230722\')',), rows_written=5)


WrittenResult(class_name='Cluster', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/cluster'), mode='merge_scoped', predicates=('target."hierarchy_id" = source."hierarchy_id" AND target."id" = source."id" AND target."hierarchy_id" IN (\'CCN20230722\')',), rows_written=6896)


WrittenResult(class_name='ClusterHierarchy', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/clusterhierarchy'), mode='merge_scoped', predicates=('target."id" = source."id"',), rows_written=1)


In [6]:
taxonomy_scope = pl.col('hierarchy_id') == HIERARCHY_ID
stored_clusters = pl.read_delta(OUTPUT_ROOT / 'cluster').filter(taxonomy_scope)
stored_categories = pl.read_delta(OUTPUT_ROOT / 'hierarchycategory').filter(taxonomy_scope)
stored_hierarchy = pl.read_delta(OUTPUT_ROOT / 'clusterhierarchy').filter(
    pl.col('id') == HIERARCHY_ID
)
stored_run = pl.read_delta(OUTPUT_ROOT / 'algorithmrun').filter(pl.col('id') == RUN_ID)
assert stored_clusters.height == stored_clusters['id'].n_unique() == 6896
assert stored_clusters['hex_color'].null_count() == 0
assert stored_categories.height == 5
assert stored_hierarchy.height == stored_run.height == 1
assert stored_hierarchy.item(0, 'root') == ROOT_ID
assert set(stored_hierarchy.row(0, named=True)['clusters']) == set(parent_of)
assert dict(zip(stored_clusters['id'], stored_clusters['parent'])) == parent_of
assert dict(zip(stored_categories['id'], stored_categories['level'])) == {
    category.id: category.level for category in categories
}
for row in stored_clusters.iter_rows(named=True):
    assert row['children'] == children_of[row['id']]
assert dict(zip(stored_clusters['id'], stored_clusters['hex_color'])) == {
    cluster.id: cluster.hex_color for cluster in clusters
}
assert json.loads(stored_run.item(0, 'json_object')) == json.loads(run.json_object)
for stored in (stored_clusters, stored_categories, stored_hierarchy, stored_run):
    print(stored.shape, stored.head(3))


(6896, 9) shape: (3, 9)
┌────────────┬────────────┬────────────┬───────┬───┬───────────┬───────────┬───────────┬───────────┐
│ id         ┆ parent     ┆ children   ┆ level ┆ … ┆ hex_color ┆ hierarchy ┆ distance_ ┆ hierarchy │
│ ---        ┆ ---        ┆ ---        ┆ ---   ┆   ┆ ---       ┆ _category ┆ to_parent ┆ _id       │
│ str        ┆ str        ┆ list[str]  ┆ i64   ┆   ┆ str       ┆ ---       ┆ ---       ┆ ---       │
│            ┆            ┆            ┆       ┆   ┆           ┆ str       ┆ f64       ┆ str       │
╞════════════╪════════════╪════════════╪═══════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ class:20   ┆ cell       ┆ ["subclass ┆ 1     ┆ … ┆ #9EF01A   ┆ class     ┆ null      ┆ CCN202307 │
│ MB GABA    ┆            ┆ :191       ┆       ┆   ┆           ┆           ┆           ┆ 22        │
│            ┆            ┆ PAG-MRN    ┆       ┆   ┆           ┆           ┆           ┆           │
│            ┆            ┆ Rln3 Ga…   ┆       ┆   ┆           ┆   

## Outputs

Paths below are relative to `OUTPUT_ROOT` and scoped to `CCN20230722`; no datasets, memberships or mappings are written.

| Path | Rows |
| --- | ---: |
| `algorithmrun/` | 1 |
| `hierarchycategory/` | 5 |
| `cluster/` | 6,896 (34 classes + 338 subclasses + 1,201 supertypes + 5,322 clusters + root) |
| `clusterhierarchy/` | 1 |

Cached inputs: `data/BARseq/extracted/cluster_to_cluster_annotation_membership_pivoted.csv` and `data/BARseq/extracted/cluster_annotation_term.csv`.